# Module 09: High-Performance Persistence & Parallel Computing with Joblib
## Notebook 04: Embarrassingly Parallel Processing for Machine Learning

Python's **GIL (Global Interpreter Lock)** prevents standard multithreading from executing CPU-bound computations concurrently across multiple cores.
`joblib.Parallel` provides a simple, pythonic, and robust multiprocessing interface that bypasses the GIL, handles inter-process serialization, and automatically optimizes batch sizes.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Master the core syntax: **`Parallel(n_jobs=...)(delayed(func)(...) for ...)`**.
2. Select the optimal backend: **`'loky'`** (process-based, robust) vs. **`'threading'`** (I/O bound).
3. Understand automatic batching (`batch_size='auto'`) and generator dispatching.
4. **Advanced:** Construct a **Parallel Cross-Validation Engine from Scratch** distributing $K$-folds across all CPU cores.
5. **Advanced:** Build a **Parallel Bootstrap Ensemble Predictor** with zero race conditions.

In [ ]:
import os
import time
import joblib
from joblib import Parallel, delayed
import numpy as np
import matplotlib.pyplot as plt

print(f"Joblib Version: {joblib.__version__}")
print(f"Available CPU Cores on System: {os.cpu_count()}")

### 1. Parallel & Delayed: Syntax and Mechanism
- **`delayed(func)`:** Wraps function calls into tuple closures `(func, args, kwargs)` without executing them immediately.
- **`Parallel(n_jobs=N)`:** Spawns a pool of worker processes, dispatches tasks, and gathers return values:
  - `n_jobs=1`: Serial sequential execution (useful for debugging).
  - `n_jobs=2`: Spawns 2 concurrent workers.
  - `n_jobs=-1`: Spawns workers on **all available CPU cores**.

In [ ]:
# Define a CPU-bound numerical task: Monte Carlo Pi estimation
def monte_carlo_pi(num_points):
    # Generates random points inside unit square and counts points inside quarter circle
    points = np.random.uniform(0.0, 1.0, (num_points, 2))
    inside_circle = np.sum(points[:, 0]**2 + points[:, 1]**2 <= 1.0)
    return inside_circle

# Benchmark Sequential vs. Parallel execution (16 iterations of 1,000,000 points)
num_tasks = 16
points_per_task = 1_000_000

# 1. Sequential execution (n_jobs=1)
t0 = time.time()
res_seq = [monte_carlo_pi(points_per_task) for _ in range(num_tasks)]
duration_seq = time.time() - t0

# 2. Parallel execution across all cores (n_jobs=-1)
t0 = time.time()
res_par = Parallel(n_jobs=-1, backend="loky")(
    delayed(monte_carlo_pi)(points_per_task) for _ in range(num_tasks)
)
duration_par = time.time() - t0

total_inside = sum(res_par)
estimated_pi = 4.0 * total_inside / (num_tasks * points_per_task)

print(f"Sequential Execution Time: {duration_seq:.2f} seconds")
print(f"Parallel Execution Time:   {duration_par:.2f} seconds")
print(f"Speedup Factor:            {duration_seq / max(duration_par, 1e-5):.2f}x")
print(f"Estimated Pi:              {estimated_pi:.5f} (True: 3.14159)")

### 2. Backends: Loky vs. Threading
- **`backend='loky'` (Default):** Robust process-based backend. Spawns isolated worker processes, safely restarts workers in case of crashes, and avoids memory leaks. **Mandatory for CPU-bound numerical code.**
- **`backend='threading'`:** Lightweight thread pool sharing memory space. Hampered by the GIL for CPU-bound tasks, but **ideal for I/O-bound operations** (HTTP requests, disk reads, downloading images).

In [ ]:
# Compare Loky vs Threading on a synthetic I/O simulation
def simulate_io_task(task_id):
    time.sleep(0.05) # Simulating network / disk read latency
    return task_id * 2

tasks = list(range(20))

t0 = time.time()
out_threads = Parallel(n_jobs=4, backend="threading")(delayed(simulate_io_task)(i) for i in tasks)
thread_time = time.time() - t0

t0 = time.time()
out_loky = Parallel(n_jobs=4, backend="loky")(delayed(simulate_io_task)(i) for i in tasks)
loky_time = time.time() - t0

print(f"I/O Task with Threading Backend: {thread_time:.3f} s (Low process spawn overhead)")
print(f"I/O Task with Loky Backend:      {loky_time:.3f} s")

### 3. Complex Application: Parallel K-Fold Cross-Validation Engine from Scratch
Scikit-Learn's `cross_val_score(..., n_jobs=-1)` is powered entirely by `joblib.Parallel`.
Below, we implement a production-grade parallel cross-validation engine from scratch:
1. Partition dataset into $K$ disjoint folds.
2. Dispatch fold training and validation tasks concurrently across CPU cores.
3. Collect out-of-fold accuracy metrics and aggregate scores with zero race conditions.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Synthesize classification dataset
X_cv, y_cv = make_classification(n_samples=1500, n_features=20, random_state=42)

def evaluate_fold(fold_id, train_idx, val_idx, X_data, y_data):
    # Worker function executing on a separate core
    X_tr, y_tr = X_data[train_idx], y_data[train_idx]
    X_va, y_va = X_data[val_idx], y_data[val_idx]

    clf = LogisticRegression(max_iter=500)
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_va)
    acc = accuracy_score(y_va, preds)
    return fold_id, acc

# Create 5 fold indices manually
K = 5
indices = np.arange(len(X_cv))
np.random.seed(42)
np.random.shuffle(indices)
fold_splits = np.array_split(indices, K)

tasks = []
for k in range(K):
    val_idx = fold_splits[k]
    train_idx = np.setdiff1d(indices, val_idx)
    tasks.append((k + 1, train_idx, val_idx))

# Execute all K folds concurrently using Joblib Parallel
print(f"Dispatching {K} Cross-Validation Folds across CPU cores...")
results = Parallel(n_jobs=-1, backend="loky")(
    delayed(evaluate_fold)(f_id, tr_idx, va_idx, X_cv, y_cv)
    for f_id, tr_idx, va_idx in tasks
)

for f_id, score in results:
    print(f"  Fold #{f_id} Accuracy: {score * 100:.2f}%")

mean_cv = np.mean([score for _, score in results])
std_cv = np.std([score for _, score in results])
print(f"\nOverall Parallel CV Score: {mean_cv * 100:.2f}% (+/- {std_cv * 100:.2f}%)")